In [ ]:
import pandas as pd
from deep_translator import GoogleTranslator
import re

In [ ]:
# load full metadata
df = pd.read_excel('merged_journals_metadata.xlsx')

In [ ]:
keywords = (df['subject']
                .dropna()
                .str.replace('; ', ' | ', regex=False)
                .str.replace('；', ' | ', regex=False)
                .str.replace(', ', ' | ', regex=False)
                .str.replace('，', ' | ', regex=False)
                .str.split(' | ', regex=False, expand=False) 
                .explode()
                .dropna()
                .str.strip('| .,;')
                .str.lower())

In [ ]:
unique_keywords = keywords.str.lower().unique().tolist()
unique_keywords = [re.sub(r'[„”«»“”]', '"', k) for k in unique_keywords]
unique_keywords = list(set(unique_keywords))
with open('unique_keywords.txt', 'w', encoding='utf-8') as txt:
    txt.writelines(line + '\n' for line in unique_keywords)
with open('to_translate.txt', 'w', encoding='utf-8') as txt:
    txt.writelines(line + '\n' for line in unique_keywords)

In [ ]:
translator = GoogleTranslator(source='auto', target='en')
with open('to_translate.txt', 'r', encoding='utf-8') as txt:
    to_translate = [e.strip() for e in txt.readlines()]
batch_size = 100
while to_translate: 
    batch = to_translate[:batch_size] 
    translated_batch = translator.translate_batch(batch)
    del to_translate[:batch_size]
    translated_merged = result = [f"{x.strip()} [--translated-->] {y.strip()}" for x, y in zip(batch, translated_batch)]
    with open('translated_keywords.txt', 'a', encoding='utf-8') as txt:
        txt.writelines(line + '\n' for line in translated_merged)
    with open('to_translate.txt', 'w', encoding='utf-8') as txt:
        txt.writelines(line + '\n' for line in to_translate)
    print('remaining to be translated -- ', len(to_translate))    